In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
batchSize = 16 # Esta parte define cuantas operaciones se pueden hacer en paralelo de forma independiente
blockSize = 64 # Define la máxima ventana de contexto para predicciones eran 32

In [3]:
#Hiperparámetros para el entrenamiento del gpt
maxIters = 8000 # más iteraciones (era 5000)
evalInterval = 100
learningRate = 5e-4 # más fino (era 1e-3)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
evalIters = 200
nEmbd = 128 # más capacidad (era 64) Numero de embeddings
nHead = 4 #Numero de Self Attention, permite que pueda ver hacia atrás 4 veces
nLayer = 4 #Numero de capas del modelo transformador
dropout = 0.1 # evita overfitting (era 0.0) No desactivamos las neuronas por probabilidades, si hay overfitting, se sube a 0.1, es decir, existe un 10% de probabilidad de que una neurona se apague.
vocabSize = 512

In [4]:
torch.manual_seed(1337) #Punto de partida del cṕdigo, no necesariamente el 1337, pueden ser otros números aleatorios. Pero los numeros aleatorios generados ahi
# Dan lo mismo.

In [5]:
import os
getPath = os.getcwd()
file = 'input.txt' #Pueden probar el dataset que gusten, recomiendo yo que ambos dataset + uno extra que ustedes busquen
newPath = os.path.join(getPath, 'dataset', file)

In [6]:
with open(newPath, 'r', encoding='utf-8') as f:
    text = f.read()

In [7]:
print(text[:100])





El ingenioso hidalgo don Quijote de la Mancha



por Miguel de Cervantes Saavedra





El ingeni


In [8]:
#Usamos nuestro tokenizador de sub splits de palabras
class BabyTokenizer:
    def __init__(self):
        self.merges = {} 
        self.vocab = {i: bytes([i]) for i in range(256)}

    def _get_stats(self, ids):
        counts = {}
        for pair in zip(ids, ids[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        return counts

    def _merge(self, ids, pair, idx):
        newids = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
                newids.append(idx)
                i += 2
            else:
                newids.append(ids[i])
                i += 1
        return newids

    def train(self, text, vocab_size, verbose=False):
        num_merges = vocab_size - 256
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)

        for i in range(num_merges):
            stats = self._get_stats(ids)
            if not stats: break
            
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = self._merge(ids, pair, idx)
            self.merges[pair] = idx
            self.vocab[idx] = self.vocab[pair[0]] + self.vocab[pair[1]]
            
            if verbose:
                print(f"Merge {i+1}/{num_merges}: {pair} -> {idx} ({self.vocab[idx].decode('utf-8', errors='replace')})")

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = self._get_stats(ids)
            pair = min(stats.keys(), key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            idx = self.merges[pair]
            ids = self._merge(ids, pair, idx)
        return ids

    def decode(self, ids):
        text_bytes = b"".join(self.vocab[idx] for idx in ids)
        return text_bytes.decode("utf-8", errors="replace")

In [9]:
tokenizer = BabyTokenizer()
trainingDataTokenizer = text
tokenizer.train(trainingDataTokenizer, vocab_size=512, verbose=False)

In [10]:
storedTokens = tokenizer.encode(text)

In [11]:
#Muestran los tokens almacenados
print(storedTokens)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [12]:
class Splitter:
    def splitt(tokens, testSize: float = 0.2):
        indexes = int(len(tokens) * (1- testSize))
        return tokens[:indexes], tokens[indexes:]

In [13]:
data = torch.tensor(storedTokens, dtype=torch.long)

In [14]:
splittzzz = Splitter
trainingData, testData = splittzzz.splitt(data, testSize=0.1)

In [15]:
print(trainingData.shape)
print(testData.shape)

torch.Size([2352663])
torch.Size([261407])


In [16]:
#No se explicó bien en clases, pero este método "divide en trozos aleatorios" dependiendo del tamaño de blocksize
def getBatch(split):
    data = trainingData if split == 'train' else testData
    ix = torch.randint(len(data) - blockSize, (batchSize,))
    x = torch.stack([data[i:i+blockSize] for i in ix])
    y = torch.stack([data[i+1:i+blockSize+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [17]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(evalIters)
        for k in range(evalIters):
            X, Y = getBatch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out



In [18]:
class Head(nn.Module):
    def __init__(self, headSize):
        super().__init__()
        self.key = nn.Linear(nEmbd, headSize, bias=False)
        self.query = nn.Linear(nEmbd, headSize, bias=False)
        self.value = nn.Linear(nEmbd, headSize, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(blockSize, blockSize)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        wei = q @ k.transpose(-2,-1) * C**-0.5 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1) 
        wei = self.dropout(wei)
        v = self.value(x) 
        out = wei @ v 
        return out


In [19]:
class MultiHeadAttention(nn.Module):
    #Multiples "cabezas" de la self-attention hechas en paralelo

    def __init__(self, numHeads, headSize):
        super().__init__()
        self.heads = nn.ModuleList([Head(headSize) for _ in range(numHeads)])
        self.proj = nn.Linear(nEmbd, nEmbd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


In [20]:
class FeedFoward(nn.Module):
    #Capa lineal el cual sigue después una parte no lineal
    def __init__(self, nEmbd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nEmbd, 4 * nEmbd),
            nn.ReLU(),
            nn.Linear(4 * nEmbd, nEmbd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [21]:
class Block(nn.Module):
    #Bloque básico del transformer, no borrar ni experimentar aquí
    def __init__(self, nEmbd, nHead):
        super().__init__()
        headSize = nEmbd // nHead
        self.sa = MultiHeadAttention(nHead, headSize)
        self.ffwd = FeedFoward(nEmbd)
        self.ln1 = nn.LayerNorm(nEmbd)
        self.ln2 = nn.LayerNorm(nEmbd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [22]:
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabSize, nEmbd)
        self.position_embedding_table = nn.Embedding(blockSize, nEmbd)
        self.blocks = nn.Sequential(*[Block(nEmbd, nHead=nHead) for _ in range(nLayer)])
        self.ln_f = nn.LayerNorm(nEmbd) 
        self.lm_head = nn.Linear(nEmbd, vocabSize)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        #NO BORREN ESTAS ANOTACIONES.
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocabSize)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, maxNewTokens):
        for _ in range(maxNewTokens):
            idx_cond = idx[:, -blockSize:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] 
            probs = F.softmax(logits, dim=-1) 
            idx_next = torch.multinomial(probs, num_samples=1) 
            idx = torch.cat((idx, idx_next), dim=1) 
        return idx



In [23]:
#SE RECOMIENDA A QUIENES TIENEN UNA GPU, USARLA, INSTALEN CUDA POR FAVOR Y LOS DRIVERS DE NVIDIA
model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'm parámetros')

optimizer = torch.optim.AdamW(model.parameters(), lr=learningRate)

print("Entrenando. Por favor espere")
for iter in range(maxIters):

    if iter % evalInterval == 0 or iter == maxIters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = getBatch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



0.931584 m parámetros
Entrenando. Por favor espere
step 0: train loss 6.4056, val loss 6.4063
step 100: train loss 5.3040, val loss 5.5713
step 200: train loss 4.8098, val loss 5.3114
step 300: train loss 4.4450, val loss 4.9352
step 400: train loss 4.2919, val loss 4.6181
step 500: train loss 4.1966, val loss 4.4661
step 600: train loss 4.1402, val loss 4.3406
step 700: train loss 4.0824, val loss 4.2694
step 800: train loss 4.0539, val loss 4.2104
step 900: train loss 4.0244, val loss 4.1343
step 1000: train loss 3.9807, val loss 4.0908
step 1100: train loss 3.9570, val loss 4.0502
step 1200: train loss 3.9268, val loss 3.9780
step 1300: train loss 3.8906, val loss 3.9775
step 1400: train loss 3.8666, val loss 3.9505
step 1500: train loss 3.8287, val loss 3.9319
step 1600: train loss 3.7955, val loss 3.8805
step 1700: train loss 3.7696, val loss 3.8376
step 1800: train loss 3.7362, val loss 3.8632
step 1900: train loss 3.7052, val loss 3.7854
step 2000: train loss 3.6671, val loss 3.

In [24]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_ids = m.generate(context, maxNewTokens=511)[0]


In [25]:
print(tokenizer.decode(generated_ids[1:].tolist()))

oneta se
que, por la tal se neriba, por tal su vida, que era la ser mañana cora;
pues sintió la comida él subirlideras de comparon doncellos tiempos
de ver había pasado con bresularlo seno, pero sí mismo de aquesta espacio. La suba,
desgalló que quizá un venído fuese Maxi alpa imponía al razón; el vigo par
y una de la casa informente habían _migo_, y Camaniguió sino de sola magua,
no había bu puundo más recesión más de Rocinante, pena
tendidas cubriendo que la esguridad a su ascolgia, de móstinancera, la
vea condistuevamente y en fanta paciencia por estaba, diectos en aquella
mesa mucha grandeza más que bus dios, y le diquerra en gente; sino Sancho que no se
estaba seguida y empezaba con soñales de segunda mañana movisión decidida,
parecioso de pertenerla cardumbre este delecimiento cleja, dijo:



«Pues quiere entendonido como tenía dura, y no sosamos los acombra; pero el
tan beso, que los viles tomar la papelita no debe que viva entrar couidio que sería
impresistión de ser arrojo, ex